# Taxi data analysis

## Tema de Interés

### Contexto
El transporte urbano de alta demanda genera volúmenes masivos de datos
operativos en tiempo real. La Ciudad de Nueva York registra millones de
viajes en taxi mensualmente a través del sistema TPEP regulado por la
Taxi & Limousine Commission (TLC).

### Objetivo General
Analizar los patrones de demanda, distribución geográfica y comportamiento
tarifario de los viajes en taxi amarillo de Nueva York (enero–marzo 2016)
mediante PySpark, con el fin de identificar variables clave que soporten
modelos predictivos de demanda y tarifa en entornos de Big Data.

### Justificación
El análisis de datos de movilidad urbana a gran escala representa un caso
de uso directo de las técnicas de Big Data, combinando procesamiento
distribuido, análisis exploratorio y preparación de datos para aplicaciones
de Machine Learning.

## Selección del Dataset

| Atributo         | Detalle |
|------------------|---------|
| **Nombre**       | NYC Yellow Taxi Trip Data |
| **Origen**       | Taxi & Limousine Commission (TLC) — NYC Government |
| **Fuente Kaggle**| https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data |
| **Fuente oficial**| https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page |
| **Período**      | Enero 2016 — Marzo 2016 |
| **Archivos**     | yellow_tripdata_2016-01.csv, 02.csv, 03.csv |
| **Tamaño total** | ~4.88 GB (34,499,859 registros, 19 columnas) |
| **Formato**      | CSV tabular |

El dataset cumple con el criterio mínimo de 1 GB y proviene de una fuente
gubernamental reconocida. Se optó por la versión de 2016 porque conserva
coordenadas geográficas reales (latitud/longitud), a diferencia de versiones
posteriores que sustituyeron las coordenadas por índices de zona (LocationID).

In [ ]:
%pip install findspark
%pip install pyspark

In [ ]:
# Repository
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==============================================================================
# CELDA 1: Import Libraries - CORRECTED VERSION
# ==============================================================================

import os
import sys
import findspark
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# Initialize Spark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, isnan, when, mean, stddev,
    min as spark_min, max as spark_max,
    to_timestamp, length, size, split,
    regexp_replace, monotonically_increasing_id,
    year, month, dayofweek, hour
)
from pyspark.sql.types import DoubleType, FloatType, IntegerType, DateType

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("LIBRARY IMPORTS SUCCESSFUL")
print("=" * 80)
print(f"Python Version: {sys.version.split()[0]}")
print(f"Spark Location: {findspark.find()}")
print("\nAll required libraries imported ✓\n")

LIBRARY IMPORTS SUCCESSFUL
Python Version: 3.12.13
Spark Location: /usr/local/lib/python3.12/dist-packages/pyspark

All required libraries imported ✓



In [ ]:
# ==============================================================================
# DATASET SOURCES — Update paths here to control which files are loaded
# ==============================================================================

base_path = "/content/drive/MyDrive/Colab Notebooks/BigData/Week02/Processed/nyc_taxi_raw"

data_path_1 = f"{base_path}/yellow_tripdata_2016-01.csv"   # January 2016
data_path_2 = f"{base_path}/yellow_tripdata_2016-02.csv"   # February 2016
data_path_3 = f"{base_path}/yellow_tripdata_2016-03.csv"   # March 2016

print("=" * 80)
print("DATASET SOURCES CONFIGURATION")
print("=" * 80)
print(f"  Dataset 1 : {data_path_1}")
print(f"  Dataset 2 : {data_path_2}")
print(f"  Dataset 3 : {data_path_3}")
print("=" * 80)
print("Update the paths above before running the notebook.")

DATASET SOURCES CONFIGURATION
  Dataset 1 : /content/drive/MyDrive/Colab Notebooks/BigData/Week02/Processed/nyc_taxi_raw/yellow_tripdata_2016-01.csv
  Dataset 2 : /content/drive/MyDrive/Colab Notebooks/BigData/Week02/Processed/nyc_taxi_raw/yellow_tripdata_2016-02.csv
  Dataset 3 : /content/drive/MyDrive/Colab Notebooks/BigData/Week02/Processed/nyc_taxi_raw/yellow_tripdata_2016-03.csv
Update the paths above before running the notebook.


In [ ]:
# ==============================================================================
# CELDA 2: Load NYC Taxi Datasets (3 sources → unified df)
# ==============================================================================

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("NYCTaxiAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# --------------------------------------------------------------------------
# Helper: load one CSV and print its summary
# --------------------------------------------------------------------------
def load_and_report(path, label):
    print(f"  Loading {label}...")
    try:
        df_tmp = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(path)
        n_records = df_tmp.count()
        n_cols    = len(df_tmp.columns)
        size_gb   = (n_records * n_cols * 8) / (1024 ** 3)
        print(f"  ✓ {label} loaded")
        print(f"    Total records   : {n_records:,}")
        print(f"    Total columns   : {n_cols}")
        print(f"    Estimated size  : ~{size_gb:.2f} GB")
        print()
        return df_tmp
    except Exception as e:
        print(f"  ✗ Error loading {label}: {e}")
        sys.exit(1)

# --------------------------------------------------------------------------
# Load individual datasets
# --------------------------------------------------------------------------
print("=" * 80)
print("LOADING DATASETS")
print("=" * 80 + "\n")

df1 = load_and_report(data_path_1, "Dataset 1 (Jan 2016)")
df2 = load_and_report(data_path_2, "Dataset 2 (Feb 2016)")
df3 = load_and_report(data_path_3, "Dataset 3 (Mar 2016)")

# --------------------------------------------------------------------------
# Combine into unified dataset — unionByName matches on column name, not position
# --------------------------------------------------------------------------
df = df1.unionByName(df2).unionByName(df3)

total_records = df.count()
total_columns = len(df.columns)
total_size_gb = (total_records * total_columns * 8) / (1024 ** 3)

print("=" * 80)
print("UNIFIED DATASET (df = df1 + df2 + df3)")
print("=" * 80)
print(f"  Total records   : {total_records:,}")
print(f"  Total columns   : {total_columns}")
print(f"  Estimated size  : ~{total_size_gb:.2f} GB")
print("=" * 80)
print("\n✓ Unified dataset ready — all subsequent cells operate on df\n")

LOADING DATASETS

  Loading Dataset 1 (Jan 2016)...
  ✓ Dataset 1 (Jan 2016) loaded
    Total records   : 10,906,858
    Total columns   : 19
    Estimated size  : ~1.54 GB

  Loading Dataset 2 (Feb 2016)...
  ✓ Dataset 2 (Feb 2016) loaded
    Total records   : 11,382,049
    Total columns   : 19
    Estimated size  : ~1.61 GB

  Loading Dataset 3 (Mar 2016)...
  ✓ Dataset 3 (Mar 2016) loaded
    Total records   : 12,210,952
    Total columns   : 19
    Estimated size  : ~1.73 GB

UNIFIED DATASET (df = df1 + df2 + df3)
  Total records   : 34,499,859
  Total columns   : 19
  Estimated size  : ~4.88 GB

✓ Unified dataset ready — all subsequent cells operate on df



In [ ]:
df.show(20)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2016-01-01 00:00:00|  2016-01-01 00:00:00|              2|          1.1|-73.99037170410156| 40.73469543457031|         1|    

In [ ]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [ ]:
# ==============================================================================
# CELDA 3: Data Structure Examination - CORRECTED
# ==============================================================================

print("\n" + "=" * 80)
print("DATA STRUCTURE ANALYSIS")
print("=" * 80 + "\n")

# 1. General Dimensions
print("1. GENERAL DIMENSIONS")
print("-" * 80)
total_records = df.count()
total_columns = len(df.columns)
print(f"Total records: {total_records:,}")
print(f"Total columns: {total_columns}")
print(f"Estimated size: ~{(total_records * total_columns * 8) / (1024**3):.2f} GB\n")

# 2. Column Names and Data Types
print("2. COLUMN NAMES & DATA TYPES")
print("-" * 80)
schema_df = pd.DataFrame({
    'Column': [field.name for field in df.schema.fields],
    'Data Type': [field.dataType.simpleString() for field in df.schema.fields]
})
print(schema_df.to_string(index=False))
print()

# 3. Descriptive Statistics
print("3. DESCRIPTIVE STATISTICS (Numeric Columns)")
print("-" * 80)
numeric_cols = [field.name for field in df.schema.fields
                if field.dataType in [DoubleType(), IntegerType()]]

if numeric_cols:
    stats = df.select(numeric_cols).describe().toPandas()
    print(stats.to_string())
    print()
else:
    print("No numeric columns found\n")

# 4. Missing Values Analysis
# isnan() only works on DOUBLE/FLOAT — use isNull() alone for all other types
print("4. MISSING VALUES ANALYSIS")
print("-" * 80)
_float_types = (DoubleType, FloatType)
null_counts = df.select([
    count(when(
        (isnan(col(c)) | col(c).isNull()) if isinstance(df.schema[c].dataType, _float_types)
        else col(c).isNull(),
        c
    )).alias(c)
    for c in df.columns
]).toPandas()

missing_df = pd.DataFrame({
    'Column': null_counts.columns,
    'Null Count': null_counts.iloc[0].values,
    'Null %': (null_counts.iloc[0].values / total_records * 100).round(2)
})
missing_df = missing_df[missing_df['Null Count'] > 0].sort_values('Null %', ascending=False)

if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
    print(f"\nColumns with missing data: {len(missing_df)}/{total_columns}")
else:
    print("No missing values detected ✓")

print()

# 5. Sample Data
print("5. SAMPLE DATA (First 5 rows)")
print("-" * 80)
sample = df.limit(5).toPandas()
print(sample.to_string(index=True))
print()

# 6. Categorical Columns Analysis
print("6. ANÁLISIS DE COLUMNAS CATEGÓRICAS")
print("-" * 80)

df.groupBy('store_and_fwd_flag').count().orderBy('count', ascending=False).show()
df.groupBy('VendorID').count().orderBy('VendorID').show()
df.groupBy('RatecodeID').count().orderBy('RatecodeID').show()
df.groupBy('payment_type').count().orderBy('payment_type').show()


DATA STRUCTURE ANALYSIS

1. GENERAL DIMENSIONS
--------------------------------------------------------------------------------
Total records: 34,499,859
Total columns: 19
Estimated size: ~4.88 GB

2. COLUMN NAMES & DATA TYPES
--------------------------------------------------------------------------------
               Column Data Type
             VendorID       int
 tpep_pickup_datetime timestamp
tpep_dropoff_datetime timestamp
      passenger_count       int
        trip_distance    double
     pickup_longitude    double
      pickup_latitude    double
           RatecodeID       int
   store_and_fwd_flag    string
    dropoff_longitude    double
     dropoff_latitude    double
         payment_type       int
          fare_amount    double
                extra    double
              mta_tax    double
           tip_amount    double
         tolls_amount    double
improvement_surcharge    double
         total_amount    double

3. DESCRIPTIVE STATISTICS (Numeric Columns)
------

In [ ]:
# ==============================================================================
# CELDA 4: Data Relevance Assessment for Research Theme
# ==============================================================================

print("\n" + "=" * 80)
print("DATA RELEVANCE ASSESSMENT")
print("=" * 80 + "\n")

print("RESEARCH THEME: Taxi Trip Patterns & Demand Prediction")
print("-" * 80 + "\n")

# Check for key features
required_features = {
    'Temporal': [
        'tpep_pickup_datetime',
        'tpep_dropoff_datetime'
    ],
    'Ubicación': [
        'pickup_latitude', 'pickup_longitude',
        'dropoff_latitude', 'dropoff_longitude'
    ],
    'Métricas de viaje': [
        'trip_distance', 'fare_amount', 'total_amount'
    ],
    'Indicadores de demanda': [
        'passenger_count', 'payment_type',
        'tip_amount', 'tolls_amount'
    ],
    'Operacionales': [
        'VendorID', 'RatecodeID',
        'store_and_fwd_flag', 'improvement_surcharge',
        'extra', 'mta_tax'
    ]
}

available_features = {}
missing_features = {}

for category, features in required_features.items():
    available = [f for f in features if f in df.columns]
    missing = [f for f in features if f not in df.columns]

    available_features[category] = available
    missing_features[category] = missing

    status = "✓" if available else "✗"
    print(f"{status} {category}:")
    if available:
        print(f"  Available: {', '.join(available)}")
    if missing:
        print(f"  Missing: {', '.join(missing)}")
    print()

# Calculate relevance score
total_required = sum(len(f) for f in required_features.values())
total_available = sum(len(f) for f in available_features.values())
relevance_score = (total_available / total_required * 100) if total_required > 0 else 0

print("-" * 80)
print(f"Data Relevance Score: {relevance_score:.1f}%")
print(f"Available features: {total_available}/{total_required}\n")

if relevance_score >= 70:
    print("✓ DATASET IS HIGHLY RELEVANT FOR THIS RESEARCH THEME")
    print("  - Contains temporal information for time-series analysis")
    print("  - Has location data for spatial patterns")
    print("  - Includes trip metrics for demand modeling")
    print("  - Sufficient for EDA, feature engineering, and ML applications")
else:
    print("⚠ DATASET HAS LIMITED RELEVANCE")
    print("  - Consider supplementing with additional data sources")


DATA RELEVANCE ASSESSMENT

RESEARCH THEME: Taxi Trip Patterns & Demand Prediction
--------------------------------------------------------------------------------

✓ Temporal:
  Available: tpep_pickup_datetime, tpep_dropoff_datetime

✓ Ubicación:
  Available: pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude

✓ Métricas de viaje:
  Available: trip_distance, fare_amount, total_amount

✓ Indicadores de demanda:
  Available: passenger_count, payment_type, tip_amount, tolls_amount

✓ Operacionales:
  Available: VendorID, RatecodeID, store_and_fwd_flag, improvement_surcharge, extra, mta_tax

--------------------------------------------------------------------------------
Data Relevance Score: 100.0%
Available features: 19/19

✓ DATASET IS HIGHLY RELEVANT FOR THIS RESEARCH THEME
  - Contains temporal information for time-series analysis
  - Has location data for spatial patterns
  - Includes trip metrics for demand modeling
  - Sufficient for EDA, feature engineering, a

In [ ]:
# ==============================================================================
# CELDA 5: Comprehensive Dataset Analysis Report - CORRECTED
# ==============================================================================

print("\n" + "=" * 80)
print("NYC TAXI DATASET - COMPREHENSIVE ANALYSIS REPORT")
print("=" * 80 + "\n")

# ============================================================================
# 1. DATASET IDENTIFICATION
# ============================================================================

print("1. DATASET IDENTIFICATION")
print("-" * 80)

dataset_info = {
    'Name': 'NYC Taxi Trip Data (Yellow Taxi)',
    'Origin': 'Taxi & Limousine Commission (TLC) - NYC Government',
    'Source URL': 'https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page',
    'Data Period': 'From January 2016 to March 2016 (3 months)',
    'Coverage': 'Yellow Taxi trips in New York City',
    'Geographic Area': 'Manhattan, Outer Boroughs, and surrounding areas'
}

for key, value in dataset_info.items():
    print(f"{key:20}: {value}")

print("\n")

# ============================================================================
# 2. GENERAL STATISTICS
# ============================================================================

print("2. GENERAL STATISTICS")
print("-" * 80)

total_records = df.count()
total_columns = len(df.columns)

stats_summary = {
    'Total Records': f"{total_records:,}",
    'Total Columns': total_columns,
    'Data Volume': f"~{(total_records * total_columns * 8) / (1024**3):.2f} GB (est.)",
    'Trips Analyzed': f"{total_records:,} individual rides"
}

for key, value in stats_summary.items():
    print(f"{key:20}: {value}")

print("\n")

# ============================================================================
# 3. MISSING VALUES & DATA QUALITY
# isnan() only works on DOUBLE/FLOAT — use isNull() alone for all other types
# ============================================================================

print("3. MISSING VALUES & DATA QUALITY")
print("-" * 80)

_float_types = (DoubleType, FloatType)
null_info = df.select([
    count(when(
        (isnan(col(c)) | col(c).isNull()) if isinstance(df.schema[c].dataType, _float_types)
        else col(c).isNull(),
        c
    )).alias(c)
    for c in df.columns
]).toPandas()

missing_summary = pd.DataFrame({
    'Column': null_info.columns,
    'Null Count': null_info.iloc[0].values,
    'Null %': (null_info.iloc[0].values / total_records * 100).round(2)
})

missing_summary = missing_summary[missing_summary['Null Count'] > 0].sort_values('Null %', ascending=False)

if len(missing_summary) > 0:
    print("Columns with missing values:\n")
    print(missing_summary.to_string(index=False))
    print(f"\nTotal columns with missing data: {len(missing_summary)}/{total_columns}")
else:
    print("✓ No missing values detected in the dataset")

print("\n")

# ============================================================================
# 4. NUMERIC DATA SUMMARY
# ============================================================================

print("4. NUMERIC DATA SUMMARY")
print("-" * 80)

numeric_cols = [field.name for field in df.schema.fields
                if field.dataType in [DoubleType(), IntegerType()]]

if numeric_cols:
    desc_stats = df.select(numeric_cols).describe().toPandas()
    print(desc_stats.to_string())
else:
    print("No numeric columns available")

print("\n")

# ============================================================================
# 5. KEY FEATURES ANALYSIS
# ============================================================================

print("5. KEY FEATURES ANALYSIS")
print("-" * 80)

# Trip distance quantiles
if 'trip_distance' in df.columns:
    try:
        dist_quantiles = df.approxQuantile('trip_distance', [0.25, 0.5, 0.75, 0.95], 0.05)
        print("\nTrip Distance (miles):")
        print(f"  Q1 (25%):     {dist_quantiles[0]:.2f}")
        print(f"  Median (50%): {dist_quantiles[1]:.2f}")
        print(f"  Q3 (75%):     {dist_quantiles[2]:.2f}")
        print(f"  P95:          {dist_quantiles[3]:.2f}")
    except:
        print("\nTrip Distance: Unable to calculate quantiles")

# Fare amount quantiles
if 'fare_amount' in df.columns:
    try:
        fare_quantiles = df.approxQuantile('fare_amount', [0.25, 0.5, 0.75, 0.95], 0.05)
        print("\nFare Amount ($):")
        print(f"  Q1 (25%):     ${fare_quantiles[0]:.2f}")
        print(f"  Median (50%): ${fare_quantiles[1]:.2f}")
        print(f"  Q3 (75%):     ${fare_quantiles[2]:.2f}")
        print(f"  P95:          ${fare_quantiles[3]:.2f}")
    except:
        print("\nFare Amount: Unable to calculate quantiles")

# Total amount quantiles
if 'total_amount' in df.columns:
    try:
        total_quantiles = df.approxQuantile('total_amount', [0.25, 0.5, 0.75, 0.95], 0.05)
        print("\nTotal Amount ($):")
        print(f"  Q1 (25%):     ${total_quantiles[0]:.2f}")
        print(f"  Median (50%): ${total_quantiles[1]:.2f}")
        print(f"  Q3 (75%):     ${total_quantiles[2]:.2f}")
        print(f"  P95:          ${total_quantiles[3]:.2f}")
    except:
        print("\nTotal Amount: Unable to calculate quantiles")

# Passenger count distribution
if 'passenger_count' in df.columns:
    try:
        pax_dist = df.groupBy('passenger_count').count().orderBy('passenger_count').toPandas()
        print("\nPassenger Count Distribution:")
        for _, row in pax_dist.iterrows():
            pct = (row['count'] / total_records * 100)
            print(f"  {int(row['passenger_count'])} passenger(s): {row['count']:,} ({pct:.1f}%)")
    except:
        print("\nPassenger Count: Unable to calculate distribution")

print("\n")

# ============================================================================
# 6. KNOWN ISSUES & CONSIDERATIONS
# ============================================================================

print("6. KNOWN ISSUES & CONSIDERATIONS")
print("-" * 80)

issues = [
    {
        'Issue': 'Coordinate Data',
        'Description': 'Some records may have missing or invalid latitude/longitude',
        'Impact': 'Use LocationID for spatial analysis if coordinates unavailable'
    },
    {
        'Issue': 'Outliers',
        'Description': 'Extremely long trips or high fares may be data entry errors',
        'Impact': 'Filter during preprocessing (e.g., trip_distance < 200)'
    },
    {
        'Issue': 'Zero Values',
        'Description': 'Some fare or distance values may be zero or negative',
        'Impact': 'Remove invalid trips before ML applications'
    },
    {
        'Issue': 'Temporal Coverage',
        'Description': 'Dataset covers specific date range; predictions need recent data',
        'Impact': 'Seasonal patterns may differ; retrain models periodically'
    },
    {
        'Issue': 'Version Format Change',
        'Description': 'The dataset author on Kaggle documents that the TLC modified its data format, replacing geographic coordinates (latitude/longitude) with location IDs. This version (2016) retains the original coordinates.',
        'Impact':'Enables geospatial clustering analysis. Not directly compatible with datasets from 2016 onwards.'
    }
]

for i, issue in enumerate(issues, 1):
    print(f"{i}. {issue['Issue']}")
    print(f"   {issue['Description']}")
    print(f"   Impact: {issue['Impact']}\n")

# ============================================================================
# 7. SUITABILITY FOR ML APPLICATIONS
# ============================================================================

print("7. SUITABILITY FOR ML APPLICATIONS")
print("-" * 80)

ml_suitability = {
    'Exploratory Data Analysis (EDA)': '***** Excellent',
    'Feature Engineering': '***** Excellent',
    'Time-Series Forecasting': '**** Good',
    'Classification': '**** Good',
    'Clustering (trip patterns)': '***** Excellent',
    'Regression (fare prediction)': '***** Excellent',
    'Deep Learning': '**** Good'
}

for task, rating in ml_suitability.items():
    print(f"{task:35}: {rating}")

print("\n" + "=" * 80)
print("✓ ANALYSIS COMPLETE")
print("=" * 80 + "\n")


NYC TAXI DATASET - COMPREHENSIVE ANALYSIS REPORT

1. DATASET IDENTIFICATION
--------------------------------------------------------------------------------
Name                : NYC Taxi Trip Data (Yellow Taxi)
Origin              : Taxi & Limousine Commission (TLC) - NYC Government
Source URL          : https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page
Data Period         : From January 2016 to March 2016 (3 months)
Coverage            : Yellow Taxi trips in New York City
Geographic Area     : Manhattan, Outer Boroughs, and surrounding areas


2. GENERAL STATISTICS
--------------------------------------------------------------------------------
Total Records       : 34,499,859
Total Columns       : 19
Data Volume         : ~4.88 GB (est.)
Trips Analyzed      : 34,499,859 individual rides


3. MISSING VALUES & DATA QUALITY
--------------------------------------------------------------------------------
✓ No missing values detected in the dataset


4. NUMERIC DATA SUMMARY


In [ ]:
# ==============================================================================
# CELDA 6: Escritura del dataset unificado en formato Parquet
# ==============================================================================

output_path = "/content/drive/MyDrive/Colab Notebooks/BigData/Week02/Processed/nyc_taxi_parquet"

print("=" * 80)
print("ESCRITURA DEL DATASET — FORMATO PARQUET")
print("=" * 80)

df.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"✓ Dataset escrito en: {output_path}")
print(f"  Registros guardados : {total_records:,}")
print(f"  Formato             : Parquet (columnar, comprimido)")
print(f"  Particiones         : {df.rdd.getNumPartitions()}")
print("=" * 80)

ESCRITURA DEL DATASET — FORMATO PARQUET
✓ Dataset escrito en: /content/drive/MyDrive/Colab Notebooks/BigData/Week02/Processed/nyc_taxi_parquet
  Registros guardados : 34,499,859
  Formato             : Parquet (columnar, comprimido)
  Particiones         : 42
